# 06 · Multi-Head Attention and the Encoder Block

**Goal:** assemble a full Transformer encoder layer: multi-head attention → residual + LayerNorm → feed-forward → residual + LayerNorm.

Concepts: *heads*, *`d_k = d_model / h`*, *output projection `W_o`*, *feed-forward network*, *residual connection*, *LayerNorm*, *parameter count*.

Standalone — needs `numpy` and `matplotlib`.

## Words you'll meet
- **Token** – one piece of the sentence (roughly a word). Like one bead on a necklace.
- **Head** – one separate attention "worker". Imagine a team of detectives where each one hunts for a different kind of clue in the same sentence.
- **Attention** – each token deciding how much to listen to every other token, like a student deciding whom to copy notes from.
- **Feed-forward network (FFN)** – a little calculator each token visits alone to think things over, after the group chat of attention.
- **Layer normalization** – re-scaling a token's numbers so they average 0 and have a typical spread of 1, like grading on a curve.
- **Residual connection** – a bypass road: the original input is added back to the result, so a layer only has to learn a small correction.
- **Mask** – a cover that hides some positions (like padding) so attention ignores them.
- **Parameters** – the adjustable numbers (the dials) that training tunes; more dials means a bigger model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

rng = np.random.default_rng(0)
B, T, d_model, h = 2, 5, 8, 2
d_k = d_model // h
x = rng.normal(size=(B, T, d_model))
print("input x:", x.shape, "  heads h =", h, "  d_k = d_model/h =", d_k)

> 🧒 **In plain words:**
> This code sets up our toy data. `softmax` turns any list of scores into percentages that add up to 1 (the biggest score gets the biggest share).
> Then we invent a tiny batch: `B = 2` examples, each with `T = 5` tokens, each token described by `d_model = 8` numbers. `x` is filled with random numbers to stand in for real word descriptions.
> We use `h = 2` heads, so each head gets `d_k = 8 / 2 = 4` numbers of each token.
> **Notice:** the printed shape `(2, 5, 8)` is exactly that: 2 examples, 5 tokens, 8 numbers each.
> **Try it:** change `h` to 4 and re-run; `d_k` should become 2.

## 1. Why several heads?
One attention pattern per layer is limiting: a token may need to look at its *previous word* **and** its *subject* **and** a *matching bracket* at the same time.
**Multi-head attention** runs `h` attentions in parallel, each in its own lower-dimensional subspace (`d_k = d_model / h`), then concatenates the results. Total cost ≈ one big head.

> **Think of it like...** a group project where one friend checks spelling, another checks facts, and a third checks the story flow. They read the same text but notice different things, then combine their notes.

## 2. Split into heads = a reshape
One big projection `X W_q` gives `(B, T, d_model)`. Reshape the last axis into `(h, d_k)` and move heads next to the batch axis:

`(B, T, d_model) → (B, T, h, d_k) → (B, h, T, d_k)`

In [ ]:
W_q = rng.normal(size=(d_model, d_model)) / np.sqrt(d_model)
Q = x @ W_q
print("Q full      :", Q.shape)

def split_heads(t, h):
    B, T, D = t.shape
    return t.reshape(B, T, h, D // h).transpose(0, 2, 1, 3)

def merge_heads(t):
    B, h, T, dk = t.shape
    return t.transpose(0, 2, 1, 3).reshape(B, T, h * dk)

Qh = split_heads(Q, h)
print("Q split     :", Qh.shape, "= (B, h, T, d_k)")
print("round trip ok:", np.allclose(merge_heads(Qh), Q))

> 🧒 **In plain words:**
> `Q = x @ W_q` multiplies every token's 8 numbers by a table of weights (`@` means matrix multiplication, a bulk 'mix the numbers with these recipes' step). The shape stays `(2, 5, 8)`.
> `split_heads` cuts each token's 8 numbers into `h` groups of `d_k`. `reshape` re-slices the same numbers into a new grid, and `transpose(0, 2, 1, 3)` swaps two axes so the head number comes right after the example number.
> `merge_heads` does the exact opposite, like cutting a loaf into slices and then putting it back together.
> **Notice:** `Q split` has shape `(2, 2, 5, 4)`: 2 examples, 2 heads, 5 tokens, 4 numbers each. `round trip ok: True` shows nothing was lost.
> **Try it:** print `Qh.shape` after changing `h` to 4 in the first cell (and re-running everything) and see the head axis grow.

Head 0 simply owns columns `0:d_k` of the vector, head 1 owns `d_k:2d_k`, and so on:

In [ ]:
print(np.allclose(Qh[:, 0], Q[:, :, :d_k]), np.allclose(Qh[:, 1], Q[:, :, d_k:]))

> 🧒 **In plain words:**
> This checks the claim above: head 0 is just the first 4 numbers of every token and head 1 is the last 4. `Qh[:, 0]` means 'all examples, head 0'; `Q[:, :, :d_k]` means 'all examples, all tokens, the first d_k numbers'.
> **Notice:** it prints `True True`, so splitting really is just cutting the list of numbers into equal chunks; no new maths happens.

## 3. Multi-head attention
Steps: project → split → attention per head (batched over `B` *and* `h`) → merge → output projection `W_o`.

In [ ]:
def multi_head_attention(x_q, x_kv, params, h, mask=None):
    W_q, W_k, W_v, W_o = params
    Q = split_heads(x_q  @ W_q, h)           # (B,h,Tq,dk)
    K = split_heads(x_kv @ W_k, h)           # (B,h,Tk,dk)
    V = split_heads(x_kv @ W_v, h)
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(d_k)   # (B,h,Tq,Tk)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    w = softmax(scores)
    ctx = merge_heads(w @ V)                  # (B,Tq,d_model)
    return ctx @ W_o, w

def init_mha(d_model, rng):
    return [rng.normal(size=(d_model, d_model)) / np.sqrt(d_model) for _ in range(4)]

mha_params = init_mha(d_model, rng)
out, w = multi_head_attention(x, x, mha_params, h)
print("output:", out.shape, " (same shape as input)")
print("weights:", w.shape, "= (B, h, T_q, T_k)")

> 🧒 **In plain words:**
> This defines the whole multi-head attention recipe. Step by step: make Q, K, V (the query, key and value versions of each token) and split them into heads.
> `scores = Q @ K.transpose(...) / np.sqrt(d_k)` compares every token's query with every token's key; a big score means 'these two are related'. Dividing by the square root of `d_k` keeps the scores from getting huge. The `transpose` just flips K so the multiplication lines up.
> `softmax` turns scores into percentages `w`. Then `w @ V` blends the values using those percentages, `merge_heads` glues the heads back, and `@ W_o` mixes the heads' answers together. `params` is simply a list of the four weight tables.
> `mask` (optional) sets blocked scores to minus infinity so softmax gives them 0%.
> **Notice:** the output shape equals the input shape `(2, 5, 8)`, while the weights are `(2, 2, 5, 5)`: for each example and each head, a 5 by 5 table saying who listens to whom.

In [ ]:
fig, axes = plt.subplots(1, h, figsize=(7, 3))
for head in range(h):
    axes[head].imshow(w[0, head], cmap="Blues", vmin=0, vmax=1)
    axes[head].set_title(f"head {head}")
    axes[head].set_xlabel("key"); axes[head].set_ylabel("query")
plt.tight_layout(); plt.show()

> 🧒 **In plain words:**
> This draws each head's 5 by 5 table for example 0 as a coloured grid. Row = the token asking (query), column = the token being looked at (key). Darker blue means more attention.
> **Notice:** the two pictures are different, which is the point: the two heads look at the sentence in different ways. Each row adds up to 1, so a dark cell means a light cell elsewhere in that row.
> **Try it:** change `w[0, head]` to `w[1, head]` to see the second example.

Different heads (different `W_q/W_k` slices) produce **different** attention patterns on the same input. After training, heads specialize.

## 4. Feed-forward network (position-wise)
After mixing information *across* tokens, each token is processed **independently** by the same 2-layer MLP:

`FFN(x) = max(0, x W1 + b1) W2 + b2` — expands to `d_ff` (paper: `4·d_model`) then back.

Attention = communication between tokens; FFN = computation within a token.

> **Think of it like...** a class discussion followed by quiet thinking time. Attention is the discussion (tokens talk to each other); the feed-forward network is each student working out their own thoughts alone.

In [ ]:
d_ff = 4 * d_model
W1 = rng.normal(size=(d_model, d_ff)) / np.sqrt(d_model); b1 = np.zeros(d_ff)
W2 = rng.normal(size=(d_ff, d_model)) / np.sqrt(d_ff);    b2 = np.zeros(d_model)

def ffn(x):
    return np.maximum(0, x @ W1 + b1) @ W2 + b2

y = ffn(x)
print("ffn:", x.shape, "->", y.shape)
print("independent per position:", np.allclose(ffn(x[:, 2:3]), y[:, 2:3]))

> 🧒 **In plain words:**
> `ffn` is a tiny two-step calculator. First `x @ W1 + b1` stretches each token from 8 numbers to `d_ff = 32` numbers (more room to think). `np.maximum(0, ...)` replaces every negative number by 0 and keeps positives, like a gate that only lets positive signals through. Then `@ W2 + b2` squeezes back down to 8.
> The second `print` runs the FFN on just token number 2 (`x[:, 2:3]`) and compares with the same token's answer from the full run.
> **Notice:** shape goes in and out as `(2, 5, 8)`, and `independent per position: True` means a token's result does not depend on its neighbours. Each token visits the calculator alone.
> **Try it:** replace the 2 with another position (for example `x[:, 4:5]` and `y[:, 4:5]`); it should still say True.

## 5. Layer normalization
For **each token vector** separately, normalize to mean 0 / variance 1 across the `d_model` features, then apply a learned scale `γ` and shift `β`. This keeps activations stable in deep stacks.

> **Think of it like...** grading on a curve: whatever the raw marks were, we re-scale them so the class average is 0 and the typical spread is 1, so later steps always see numbers of a similar size.

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return gamma * (x - mu) / np.sqrt(var + eps) + beta

gamma, beta = np.ones(d_model), np.zeros(d_model)
big = rng.normal(5, 3, size=(B, T, d_model))          # badly scaled activations
n = layer_norm(big, gamma, beta)
print("before: mean %.2f  std %.2f" % (big.mean(), big.std()))
print("after : per-token mean ~", n.mean(-1).round(6).ravel()[:3], " std ~", n.std(-1).round(3).ravel()[:3])

> 🧒 **In plain words:**
> `layer_norm` handles each token on its own. `x.mean(axis=-1)` and `x.var(axis=-1)` compute the average and the spread (variance) across that token's 8 numbers; `axis=-1` means 'along the last axis', i.e. within each token. Then it subtracts the average and divides by the spread, and finally `gamma` and `beta` let the model stretch and shift the result (here they are 1 and 0, so they change nothing yet). `eps` is a tiny number that avoids dividing by zero.
> We feed in deliberately messy numbers (average about 5, big spread) and clean them.
> **Notice:** before, the mean is about 4 and the std (typical spread) is about 2.85. After, each token has mean about 0 and std 1, so every token is on the same scale, like grading on a curve.
> **Try it:** change `rng.normal(5, 3, ...)` to `rng.normal(100, 50, ...)`; the 'after' line should still show 0 and 1.

## 6. Residual (skip) connections
Each sub-layer is wrapped as `LayerNorm(x + Sublayer(x))` (the paper's "post-norm"). The `+ x` gives gradients a direct path through deep stacks and means a sub-layer only has to learn a *correction*.

> **Think of it like...** a bypass road next to a busy town. Traffic (the original information) can always go straight through, and the town (the sub-layer) only adds its own small detour. It is also like editing an essay: you keep your draft and only write the corrections.

## 7. One encoder layer
```
x ─► MHA(self) ─► +x ─► LayerNorm ─► FFN ─► +  ─► LayerNorm ─► out
```

In [ ]:
def init_encoder_layer(d_model, d_ff, rng):
    return {
        "mha": init_mha(d_model, rng),
        "W1": rng.normal(size=(d_model, d_ff)) / np.sqrt(d_model), "b1": np.zeros(d_ff),
        "W2": rng.normal(size=(d_ff, d_model)) / np.sqrt(d_ff),    "b2": np.zeros(d_model),
        "g1": np.ones(d_model), "be1": np.zeros(d_model),
        "g2": np.ones(d_model), "be2": np.zeros(d_model),
    }

def encoder_layer(x, p, h, pad_mask=None):
    mask = None if pad_mask is None else pad_mask[:, None, None, :]   # (B,1,1,Tk)
    attn, w = multi_head_attention(x, x, p["mha"], h, mask)
    x = layer_norm(x + attn, p["g1"], p["be1"])
    f = np.maximum(0, x @ p["W1"] + p["b1"]) @ p["W2"] + p["b2"]
    x = layer_norm(x + f, p["g2"], p["be2"])
    return x, w

layer = init_encoder_layer(d_model, d_ff, rng)
out, w = encoder_layer(x, layer, h)
print("in", x.shape, "-> out", out.shape)

> 🧒 **In plain words:**
> `init_encoder_layer` makes a dictionary (a labelled box of settings): `mha` holds the attention tables, `W1, b1, W2, b2` belong to the FFN, and `g1, be1, g2, be2` are the scale and shift of the two layer norms. Using names like `p["W1"]` means 'open the box and take the table labelled W1'.
> `encoder_layer` then does the recipe in order: attention, add the original `x` back (the bypass road), normalise; then FFN, add back again, normalise. The padding mask is reshaped with `[:, None, None, :]` so it can be applied to every head and every query.
> **Notice:** the input and output shapes are both `(2, 5, 8)`, so layers can be chained.

### With a padding mask
Pretend the last two tokens of example 1 are `<PAD>`:

In [ ]:
pad_mask = np.ones((B, T), dtype=bool)       # True = real token
pad_mask[1, 3:] = False
out, w = encoder_layer(x, layer, h, pad_mask)
print("attention paid to padded keys (example 1):", w[1, :, :, 3:].max())

> 🧒 **In plain words:**
> `pad_mask` is a table of True/False: True means 'real token', False means 'filler'. We mark tokens 3 and 4 of example 1 as filler (`pad_mask[1, 3:] = False`). Then we look at how much attention was paid to those filler columns (`w[1, :, :, 3:]`) and take the biggest value.
> **Notice:** the printed value is 0.0, so nobody listens to the filler, like ignoring blank pages at the end of a notebook.
> **Try it:** run the same line with `pad_mask` removed (`encoder_layer(x, layer, h)`) and the number should be above zero.

## 8. Stack N layers
Shape never changes, so layers simply chain: `(B, T, d_model) → … → (B, T, d_model)`. The paper uses `N = 6`.

In [ ]:
N = 3
layers = [init_encoder_layer(d_model, d_ff, rng) for _ in range(N)]
z = x
for i, p in enumerate(layers):
    z, _ = encoder_layer(z, p, h)
    print(f"after layer {i+1}: shape {z.shape}  mean {z.mean():+.3f}  std {z.std():.3f}")

> 🧒 **In plain words:**
> This builds `N = 3` separate layers (each with its own random weights) and passes the data through them one after another, in a loop. `z` is the data's current state. `z, _ =` keeps the result and ignores the attention table.
> **Notice:** the shape stays `(2, 5, 8)` after every layer, and the mean is about 0 with std 1.000 each time, because every layer ends with a layer norm. That stability is why deep stacks work.
> **Try it:** set `N = 6` (the paper's choice); it should still look stable.

## 9. Where do the parameters live?
Per encoder layer: attention `4·d²` (+0 biases here) + FFN `2·d·d_ff + d_ff + d` + LayerNorm `4·d`.

In [ ]:
def count(p):
    n = sum(a.size for a in p["mha"])
    n += sum(v.size for k, v in p.items() if k != "mha")
    return n

d, f = d_model, d_ff
formula = 4*d*d + (2*d*f + f + d) + 4*d
print("counted :", count(layers[0]))
print("formula :", formula)

# paper-sized layer: d=512, d_ff=2048
d, f = 512, 2048
print(f"\npaper-size encoder layer ≈ {4*d*d + 2*d*f + f + d + 4*d:,} parameters; x6 = {6*(4*d*d + 2*d*f + f + d + 4*d):,}")

> 🧒 **In plain words:**
> `count` adds up how many numbers (parameters, the dials that training adjusts) a layer holds: `a.size` is the number of entries in a table. The `formula` line gets the same total by pure arithmetic.
> **Notice:** both print 840, so our hand formula is right. For the paper's real sizes (d=512, d_ff=2048), one layer has about 3.15 million parameters, and six layers about 18.9 million.
> **Try it:** change `d_model` to 16 in the first cell and re-run everything; the counted and formula numbers should still match, but be larger.

## Recap
- Multi-head = split `d_model` into `h` chunks, attend in each, concatenate, project with `W_o`.
- Encoder layer = **MHA → Add&Norm → FFN → Add&Norm**; input and output shapes are identical.
- Attention mixes across positions; FFN transforms each position.

➡ Next: **07 · Decoder, Loss and Generation**.